# UKF–SIR Narrative Model
## Day 2: Fake Data Test (Single Observable — Google Trends)

### State-Space Model

**Hidden state**: $\mathbf{x}_t = [S_t, I_t]^\top$ (susceptible, infected — narrative-engaged)

**Transition (Euler, $\Delta t = 1$ day)**:
$$S_{t+1} = S_t - \beta S_t I_t + w_S$$
$$I_{t+1} = I_t + (\beta S_t I_t - \gamma I_t) + w_I$$
$$\mathbf{w}_t \sim \mathcal{N}(\mathbf{0}, Q)$$

**Observation (GT only, $c_1 = 1$ fixed)**:
$$y_t^{GT} = I_t + v_t, \qquad v_t \sim \mathcal{N}(0, R_{GT})$$

**Parameters to estimate**: $\theta = [\beta, \gamma]$

**Goal (Day 2)**: Simulate with known $\theta^*$, add noise, run UKF + MLE, verify recovery.

In [ ]:
import numpy as np
from scipy.linalg import cholesky
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

---
## Day 1 — UKF Infrastructure

### SIR Transition Function

In [ ]:
def sir_transition(state: np.ndarray, beta: float, gamma: float, dt: float = 1.0) -> np.ndarray:
    """Euler-discretised SIR step. Compartments normalised so S+I+R=1."""
    S, I = np.clip(state[0], 0.0, 1.0), np.clip(state[1], 0.0, 1.0)
    S_new = np.clip(S - beta * S * I * dt,            0.0, 1.0)
    I_new = np.clip(I + (beta * S * I - gamma * I) * dt, 0.0, 1.0)
    return np.array([S_new, I_new])

### Observation Function (GT, $c_1 = 1$)

In [ ]:
def obs_gt(state: np.ndarray) -> np.ndarray:
    """y = c1 * I,  c1 = 1 fixed."""
    return np.array([state[1]])

### Merwe Scaled Sigma Points

In [ ]:
def _sigma_points(
    x: np.ndarray,
    P: np.ndarray,
    alpha: float = 1e-3,
    kappa: float = 0.0,
    beta_ukf: float = 2.0,
):
    """
    Returns sigma points and weights for the Merwe scaled transform.
    
    Returns
    -------
    sigmas : (2n+1, n)
    Wm     : (2n+1,)  mean weights
    Wc     : (2n+1,)  covariance weights
    """
    n = len(x)
    lam = alpha**2 * (n + kappa) - n

    Wm = np.full(2 * n + 1, 0.5 / (n + lam))
    Wc = np.full(2 * n + 1, 0.5 / (n + lam))
    Wm[0] = lam / (n + lam)
    Wc[0] = lam / (n + lam) + (1.0 - alpha**2 + beta_ukf)

    # sqrt((n+lam)*P) via Cholesky
    P_safe = P + 1e-10 * np.eye(n)          # mild regularisation
    L = cholesky((n + lam) * P_safe, lower=True)

    sigmas = np.empty((2 * n + 1, n))
    sigmas[0] = x
    for i in range(n):
        sigmas[i + 1]     = x + L[:, i]
        sigmas[n + i + 1] = x - L[:, i]

    return sigmas, Wm, Wc

### UKF Predict & Update

In [ ]:
def ukf_predict(x, P, beta, gamma, Q, dt=1.0):
    """
    UKF prediction step.

    Returns
    -------
    x_pred  : (n,)      predicted state mean
    P_pred  : (n,n)     predicted state covariance
    sigmas_f: (2n+1, n) propagated sigma points
    Wm, Wc  : weights (passed forward to update step)
    """
    n = len(x)
    sigmas, Wm, Wc = _sigma_points(x, P)

    sigmas_f = np.array([sir_transition(s, beta, gamma, dt) for s in sigmas])

    x_pred = Wm @ sigmas_f                    # (n,)

    P_pred = Q.copy()
    for i in range(2 * n + 1):
        d = sigmas_f[i] - x_pred
        P_pred += Wc[i] * np.outer(d, d)

    return x_pred, P_pred, sigmas_f, Wm, Wc


def ukf_update(x_pred, P_pred, sigmas_f, Wm, Wc, y_obs, h_func, R):
    """
    UKF measurement update step.

    Returns
    -------
    x_upd  : updated state mean
    P_upd  : updated state covariance
    innov  : innovation vector (y_obs - y_pred)
    S_yy   : innovation covariance
    """
    n = len(x_pred)
    sigmas_h = np.array([h_func(s) for s in sigmas_f])   # (2n+1, m)
    m = sigmas_h.shape[1]

    y_pred = Wm @ sigmas_h                    # (m,)

    S_yy = R.copy()
    P_xy = np.zeros((n, m))
    for i in range(2 * n + 1):
        dy = sigmas_h[i] - y_pred
        dx = sigmas_f[i] - x_pred
        S_yy += Wc[i] * np.outer(dy, dy)
        P_xy += Wc[i] * np.outer(dx, dy)

    K     = P_xy @ np.linalg.inv(S_yy)       # Kalman gain (n, m)
    innov = y_obs - y_pred

    x_upd = x_pred + K @ innov
    P_upd = P_pred - K @ S_yy @ K.T
    P_upd = 0.5 * (P_upd + P_upd.T)          # enforce symmetry

    return x_upd, P_upd, innov, S_yy

### Log-Likelihood & Optimizer

In [ ]:
def ukf_loglik(beta, gamma, y_gt, x0, P0, Q, R_gt, dt=1.0):
    """
    Run UKF over the full time series and return the sum of Gaussian
    log-likelihoods of the innovations:

        log p(y_{1:T} | θ) = Σ_t  -½ [ log|S_t| + νt' S_t⁻¹ νt + m log(2π) ]

    Returns -∞ (as -1e10) if a covariance becomes degenerate.
    """
    x, P = x0.copy(), P0.copy()
    ll   = 0.0

    for t in range(len(y_gt)):
        x_pred, P_pred, sigmas_f, Wm, Wc = ukf_predict(x, P, beta, gamma, Q, dt)
        y_obs = np.array([y_gt[t]])
        x, P, innov, S_yy = ukf_update(x_pred, P_pred, sigmas_f, Wm, Wc,
                                         y_obs, obs_gt, R_gt)

        sign, logdet = np.linalg.slogdet(S_yy)
        if sign <= 0:
            return -1e10
        m   = len(innov)
        ll += -0.5 * (logdet + innov @ np.linalg.solve(S_yy, innov)
                      + m * np.log(2.0 * np.pi))

    return ll


def estimate_params(y_gt, x0, P0, Q, R_gt, beta_init=0.3, gamma_init=0.1):
    """
    MLE via Nelder-Mead in log-parameter space to enforce positivity.
    Returns (beta_est, gamma_est, scipy OptimizeResult)
    """
    def neg_ll(theta):
        b, g = np.exp(theta[0]), np.exp(theta[1])
        return -ukf_loglik(b, g, y_gt, x0, P0, Q, R_gt)

    theta0 = [np.log(beta_init), np.log(gamma_init)]
    result = minimize(neg_ll, theta0, method='Nelder-Mead',
                      options={'maxiter': 10_000, 'xatol': 1e-7, 'fatol': 1e-7})
    return np.exp(result.x[0]), np.exp(result.x[1]), result

---
## Day 2 — Fake Data Test

1. Fix true parameters $\beta^* = 0.30$, $\gamma^* = 0.10$.
2. Simulate 100-day SIR trajectory with process noise.
3. Generate noisy GT observations $y^{GT}_t = I_t + v_t$.
4. Run UKF + MLE with deliberately misspecified starting point.
5. Verify recovery.

In [ ]:
# ── True parameters ────────────────────────────────────────────────────────
BETA_TRUE  = 0.30    # narrative spread rate  (β*)
GAMMA_TRUE = 0.10    # recovery rate           (γ*)
T          = 100     # simulation length (days)
dt         = 1.0

SIGMA_Q = 0.005      # process noise std  (same for S and I)
SIGMA_R = 0.020      # GT observation noise std

rng = np.random.default_rng(seed=42)

# ── Simulate true S, I trajectory ─────────────────────────────────────────
Q_sim  = np.diag([SIGMA_Q**2, SIGMA_Q**2])
S_true = np.zeros(T)
I_true = np.zeros(T)
S_true[0], I_true[0] = 0.99, 0.01

for t in range(1, T):
    nxt  = sir_transition(np.array([S_true[t-1], I_true[t-1]]),
                          BETA_TRUE, GAMMA_TRUE, dt)
    w    = rng.multivariate_normal([0.0, 0.0], Q_sim)
    S_true[t] = np.clip(nxt[0] + w[0], 0.0, 1.0)
    I_true[t] = np.clip(nxt[1] + w[1], 0.0, 1.0)

# ── Generate GT observations ───────────────────────────────────────────────
y_gt = np.clip(I_true + rng.normal(0.0, SIGMA_R, T), 0.0, None)

print(f"I_true range : [{I_true.min():.4f}, {I_true.max():.4f}]")
print(f"y_gt   range : [{y_gt.min():.4f},  {y_gt.max():.4f}]")

In [ ]:
# ── UKF / filter settings ──────────────────────────────────────────────────
Q_filt = np.diag([SIGMA_Q**2, SIGMA_Q**2])
R_gt   = np.array([[SIGMA_R**2]])

x0 = np.array([0.99, 0.01])          # filter initial state
P0 = np.diag([1e-4, 1e-4])           # filter initial covariance

# ── Parameter estimation ───────────────────────────────────────────────────
# Start far from truth so recovery is non-trivial
beta_est, gamma_est, opt = estimate_params(
    y_gt, x0, P0, Q_filt, R_gt,
    beta_init=0.15, gamma_init=0.20,
)

print("\n=== Parameter Recovery ===")
print(f"  True   β = {BETA_TRUE:.4f}   γ = {GAMMA_TRUE:.4f}")
print(f"  Est.   β = {beta_est:.4f}   γ = {gamma_est:.4f}")
print(f"  β err  = {abs(beta_est - BETA_TRUE)/BETA_TRUE*100:.2f}%")
print(f"  γ err  = {abs(gamma_est - GAMMA_TRUE)/GAMMA_TRUE*100:.2f}%")
print(f"  Optimizer: success={opt.success}, iterations={opt.nit}")

In [ ]:
# ── Run UKF with estimated params to get filtered trajectory ──────────────
def run_ukf_full(beta, gamma, y_gt, x0, P0, Q, R_gt, dt=1.0):
    T = len(y_gt)
    x_hist     = np.zeros((T, 2))
    P_hist     = np.zeros((T, 2, 2))
    innov_hist = np.zeros(T)
    Syy_hist   = np.zeros(T)

    x, P = x0.copy(), P0.copy()
    for t in range(T):
        x_pred, P_pred, sigmas_f, Wm, Wc = ukf_predict(x, P, beta, gamma, Q, dt)
        x, P, innov, S_yy = ukf_update(x_pred, P_pred, sigmas_f, Wm, Wc,
                                         np.array([y_gt[t]]), obs_gt, R_gt)
        x_hist[t]     = x
        P_hist[t]     = P
        innov_hist[t] = innov[0]
        Syy_hist[t]   = S_yy[0, 0]

    return x_hist, P_hist, innov_hist, Syy_hist


x_filt, P_filt, innov, Syy = run_ukf_full(
    beta_est, gamma_est, y_gt, x0, P0, Q_filt, R_gt
)

# Standardised innovations (should be ~N(0,1) if model is correct)
std_innov = innov / np.sqrt(Syy)

print(f"Standardised innovation mean : {std_innov.mean():.4f}  (expected ≈ 0)")
print(f"Standardised innovation std  : {std_innov.std():.4f}  (expected ≈ 1)")

In [ ]:
# ── Plots ──────────────────────────────────────────────────────────────────
days  = np.arange(T)
s_S   = np.sqrt(P_filt[:, 0, 0])
s_I   = np.sqrt(P_filt[:, 1, 1])

fig = plt.figure(figsize=(13, 10))
gs  = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.35)
fig.suptitle(
    f'Day 2 — UKF Parameter Recovery | '
    f'True (β={BETA_TRUE}, γ={GAMMA_TRUE})  →  Est (β={beta_est:.3f}, γ={gamma_est:.3f})',
    fontsize=12, fontweight='bold'
)

# ─ S trajectory ─
ax = fig.add_subplot(gs[0, 0])
ax.plot(days, S_true, 'steelblue', lw=1.8, label='True $S$')
ax.plot(days, x_filt[:, 0], 'tomato', lw=1.5, ls='--', label='UKF $\\hat{S}$')
ax.fill_between(days,
    x_filt[:, 0] - 2*s_S, x_filt[:, 0] + 2*s_S,
    color='tomato', alpha=0.18, label='±2σ')
ax.set(xlabel='Day', ylabel='S', title='Susceptible')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ─ I trajectory + GT obs ─
ax = fig.add_subplot(gs[0, 1])
ax.scatter(days, y_gt, s=8, c='grey', alpha=0.5, zorder=1, label='GT obs')
ax.plot(days, I_true, 'steelblue', lw=1.8, label='True $I$', zorder=2)
ax.plot(days, x_filt[:, 1], 'tomato', lw=1.5, ls='--', label='UKF $\\hat{I}$', zorder=3)
ax.fill_between(days,
    x_filt[:, 1] - 2*s_I, x_filt[:, 1] + 2*s_I,
    color='tomato', alpha=0.18, label='±2σ')
ax.set(xlabel='Day', ylabel='I', title='Infected / Narrative-Engaged')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ─ Standardised innovations ─
ax = fig.add_subplot(gs[1, 0])
ax.plot(days, std_innov, 'k', lw=0.8)
ax.axhline(0,  color='red',  ls='--', lw=1)
ax.axhline( 1.96, color='orange', ls=':', lw=1, label='±1.96')
ax.axhline(-1.96, color='orange', ls=':', lw=1)
ax.set(xlabel='Day', ylabel='Std. innovation', title='Standardised Innovations (should ≈ N(0,1))')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ─ Innovation histogram ─
ax = fig.add_subplot(gs[1, 1])
ax.hist(std_innov, bins=15, density=True, color='steelblue', alpha=0.7, label='Empirical')
z = np.linspace(-4, 4, 200)
ax.plot(z, np.exp(-0.5*z**2)/np.sqrt(2*np.pi), 'r-', lw=1.5, label='N(0,1)')
ax.set(xlabel='Std. innovation', ylabel='Density', title='Innovation Histogram')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ─ Parameter bar chart ─
ax = fig.add_subplot(gs[2, :])
labels = ['β (spread rate)', 'γ (recovery rate)']
true_v = [BETA_TRUE,  GAMMA_TRUE]
est_v  = [beta_est,   gamma_est]
x_pos  = np.array([0.0, 1.0])
bars_t = ax.bar(x_pos - 0.18, true_v, 0.32, label='True', color='steelblue', alpha=0.8)
bars_e = ax.bar(x_pos + 0.18, est_v,  0.32, label='Estimated', color='tomato', alpha=0.8)
for bar, tv, ev in zip(bars_e, true_v, est_v):
    err = abs(ev - tv) / tv * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{err:.1f}%', ha='center', va='bottom', fontsize=9, color='tomato')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=11)
ax.set(ylabel='Value', title='Parameter Recovery (% = relative error)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.savefig('day2_ukf_recovery.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → day2_ukf_recovery.png')

---
### Summary

| Quantity | True | Estimated |
|---|---|---|
| β (spread rate) | 0.30 | (see output) |
| γ (recovery rate) | 0.10 | (see output) |

**What to check before moving to Day 3:**
- Both parameters recovered within ~5 % of truth.
- Standardised innovations have mean ≈ 0 and std ≈ 1.
- Histogram roughly matches N(0,1).

**Next (Day 3):** Add Articles and Tone observables with free scaling parameters $c_2, c_3 < 0$ and implement an observation mask for days where GDELT data is missing.